# Beam Selection in a Mobile mmWave Link: Strongest-RSRP Baseline vs MLP

**Assignment:** Chapter 2 design assignment, option 1 (supervised beam selection)

**Problem.** A mmWave base station (32-antenna array, 32-beam DFT codebook) serves a moving user. Before each transmission it must pick one beam. The measurements are one sweep old (mobility makes them stale), noisy, and the LoS path is sometimes blocked.

| Item | Choice |
|---|---|
| Inputs (all measurable before transmission) | RSRP of the swept beams (one sweep old), noisy user position, noisy velocity |
| Output | best beam index (32-class classification) |
| Baseline | beam with the strongest measured RSRP (trusted operational rule) |
| ML method | MLP classifier |
| Task metrics | top-1 / top-3 accuracy |
| Wireless metrics | SNR loss vs the best beam, outage (>3 dB loss), spectral efficiency |
| Cost metrics | beams measured, inference time |
| Leakage prevention | split by **complete route** (no route appears in two splits); preprocessing fitted on training routes only |
| Robustness | per speed bin, per SNR bin, LoS vs blocked, 3 random seeds |

Run all cells in order (Runtime > Run all). Runs on CPU in a few minutes.

In [ ]:
import time, copy, sys, platform
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler

# ---------------- Experiment configuration (everything that defines the result) ----------------
SEEDS      = [0, 1, 2]       # dataset + model seeds (results are reported as mean +/- std)
M          = 32              # BS antennas (half-wavelength ULA)
B          = 32              # codebook size (DFT beams)
STRIDE     = 1               # measure every STRIDE-th beam (1 = full sweep; 4 = partial sweep)
N_ROUTES   = 500             # user routes per dataset
T          = 50              # time steps per route
DT         = 0.1             # seconds between sweeps / time steps
SPEED_RANGE = (1.0, 25.0)    # m/s (walking ... vehicle)
PTX_RANGE  = (0.0, 30.0)     # dBm, drawn per route (sets the SNR condition)
P_BLOCK    = 0.5             # probability that a route has a LoS blockage interval
BLOCK_LOSS = 25.0            # dB extra loss on the LoS path while blocked
RICIAN_K   = 10.0            # dB, LoS fading
NLOS_LOSS  = 15.0            # dB reflection loss on scattered paths
FC         = 28e9            # carrier frequency (Hz)
NOISE_DBM  = -174 + 10*np.log10(100e6) + 7   # 100 MHz bandwidth, 7 dB noise figure
POS_STD, VEL_STD = 1.5, 0.5  # measurement noise on position (m) and velocity (m/s)
EPOCHS     = 40
SCATTERERS = np.array([[-45., 70.], [40., 45.], [0., 105.]])   # fixed reflectors
AREA_X, AREA_Y = (-50., 50.), (20., 100.)                        # BS at the origin

print("python", platform.python_version(), "| numpy", np.__version__, "| sklearn", sklearn.__version__)

## 1. Wireless scenario and data generation

In [ ]:
LAMBDA = 3e8 / FC
m_idx  = np.arange(M)
s_beam = -1 + (2*np.arange(B) + 1) / B                  # beam centres in sin(angle) space
W      = np.exp(1j*np.pi*np.outer(m_idx, s_beam)) / np.sqrt(M)   # (M, B) codebook
meas_idx = np.arange(0, B, STRIDE)                        # beams that are actually measured

def cn(shape, rng):
    return (rng.standard_normal(shape) + 1j*rng.standard_normal(shape)) / np.sqrt(2)

def fspl_db(d):
    return 20*np.log10(4*np.pi*d/LAMBDA)

def gen_routes(R, rng):
    """User routes: constant speed, gently changing heading, reflecting at the area edges."""
    speed = rng.uniform(*SPEED_RANGE, R)
    pos   = np.stack([rng.uniform(-40, 40, R), rng.uniform(30, 90, R)], 1)
    head  = rng.uniform(0, 2*np.pi, R)
    P = np.zeros((R, T, 2))
    for t in range(T):
        P[:, t] = pos
        head = head + rng.normal(0, 0.15, R)
        pos = pos + speed[:, None]*np.stack([np.cos(head), np.sin(head)], 1)*DT
        for k, (lo, hi) in enumerate([AREA_X, AREA_Y]):
            below, above = pos[:, k] < lo, pos[:, k] > hi
            pos[below, k] = 2*lo - pos[below, k]
            pos[above, k] = 2*hi - pos[above, k]
            flip = below | above
            head[flip] = (np.pi - head[flip]) if k == 0 else (-head[flip])
    return P, speed

def simulate(R, rng):
    """Generate R routes -> beam-selection dataset. Channel amplitudes are in SNR units (noise power = 1)."""
    P, speed = gen_routes(R, rng)
    ptx = rng.uniform(*PTX_RANGE, R)[:, None]                          # dBm per route
    has = rng.random(R) < P_BLOCK
    start = rng.integers(0, T-20, R); length = rng.integers(8, 21, R)
    tt = np.arange(T)[None]
    blocked = has[:, None] & (tt >= start[:, None]) & (tt < (start+length)[:, None])   # (R, T)

    def add_path(h, s_in, pl_db, los):
        amp = np.sqrt(10**((ptx - pl_db - NOISE_DBM)/10))             # (R, T)
        if los:
            k = 10**(RICIAN_K/10)
            g = np.sqrt(k/(k+1))*np.exp(1j*rng.uniform(0, 2*np.pi, amp.shape)) + np.sqrt(1/(k+1))*cn(amp.shape, rng)
        else:
            g = cn(amp.shape, rng)
        a = np.exp(1j*np.pi*m_idx*s_in[..., None])                      # steering vector (R, T, M)
        return h + (amp*g)[..., None]*a

    d = np.linalg.norm(P, axis=-1)
    h = np.zeros((R, T, M), complex)
    h = add_path(h, P[..., 0]/d, fspl_db(d) + BLOCK_LOSS*blocked, los=True)       # LoS (maybe blocked)
    for sc in SCATTERERS:                                                           # NLoS via fixed reflectors
        d_sc = np.linalg.norm(sc)
        d_tot = d_sc + np.linalg.norm(P - sc, axis=-1)
        h = add_path(h, np.full((R, T), sc[0]/d_sc), fspl_db(d_tot) + NLOS_LOSS, los=False)

    z = h @ W.conj()                                   # (R, T, B) beamformed channel w_b^H h
    snr_true = np.abs(z)**2                            # achieved SNR of each beam (linear)
    rsrp = np.abs(z + cn(z.shape, rng))**2             # what the receiver actually measures (noisy)
    best = snr_true.argmax(-1)                         # label: best beam at each time step

    # ---- build samples: features come from time t-1 (one sweep old), target is the beam at time t ----
    X, y, meta = [], [], {k: [] for k in ["route", "speed", "blocked", "snr_true", "rsrp"]}
    rsrp_db = np.clip(10*np.log10(rsrp + 1e-2), -20, 60)
    pos_n = P + rng.normal(0, POS_STD, P.shape)
    vel_n = np.zeros_like(P); vel_n[:, 1:] = (P[:, 1:] - P[:, :-1])/DT
    vel_n = vel_n + rng.normal(0, VEL_STD, P.shape)
    for t in range(2, T):
        feats = np.concatenate([rsrp_db[:, t-1][:, meas_idx],            # beam RSRP (dB)
                                pos_n[:, t-1], vel_n[:, t-1]], axis=1)   # position (x,y), velocity (vx,vy)
        X.append(feats); y.append(best[:, t])
        meta["route"].append(np.arange(R)); meta["speed"].append(speed)
        meta["blocked"].append(blocked[:, t]); meta["snr_true"].append(snr_true[:, t])
        meta["rsrp"].append(rsrp[:, t-1][:, meas_idx])
    D = {k: np.concatenate(v) for k, v in meta.items()}
    D["X"] = np.concatenate(X); D["y"] = np.concatenate(y)
    return D

rng = np.random.default_rng(0)
D = simulate(N_ROUTES, rng)
best_snr_db = 10*np.log10(D["snr_true"].max(1))
print("samples:", len(D["y"]), "| feature dim:", D["X"].shape[1], "| beams measured:", len(meas_idx))
print("best-beam SNR (dB) percentiles 5/25/50/75/95:", np.percentile(best_snr_db, [5, 25, 50, 75, 95]).round(1))
print("fraction of samples with blocked LoS:", D["blocked"].mean().round(2))

## 2. Leakage-safe split, baseline and metrics

* Samples inside one route are almost identical neighbours, so a random sample split leaks information. The split is made **by route**, and the notebook checks that no route appears in two splits.
* The scaler is fitted on training routes only.
* The baseline uses exactly the same measurements as the MLP.

In [ ]:
def make_splits(route, rng, mode="route"):
    n_routes = route.max() + 1
    if mode == "route":
        perm = rng.permutation(n_routes)
        ntr, nva = int(.6*n_routes), int(.2*n_routes)
        grp = np.empty(n_routes, int)
        grp[perm[:ntr]] = 0; grp[perm[ntr:ntr+nva]] = 1; grp[perm[ntr+nva:]] = 2
        g = grp[route]
    else:                                                     # random sample split (leaky, used only for the demo)
        g = rng.choice([0, 1, 2], size=len(route), p=[.6, .2, .2])
    return g == 0, g == 1, g == 2

def baseline_strongest_rsrp(rsrp):
    order = np.argsort(-rsrp, axis=1)[:, :3]
    return meas_idx[order[:, 0]], meas_idx[order]            # top-1 and top-3 candidates

def evaluate(pred1, cand3, snr_true, y):
    n = len(y); idx = np.arange(n)
    best = snr_true[idx, y]
    s1 = snr_true[idx, pred1]
    s3 = snr_true[idx[:, None], cand3].max(1)                 # probe the 3 candidates, keep the best
    loss1 = 10*np.log10(best/s1); loss3 = 10*np.log10(best/s3)
    return {"top1": np.mean(pred1 == y), "top3": np.mean((cand3 == y[:, None]).any(1)),
            "within1": np.mean(np.abs(pred1 - y) <= 1),
            "snr_loss_dB": loss1.mean(), "snr_loss_top3_dB": loss3.mean(),
            "outage_%": 100*np.mean(loss1 > 3), "rate_bps_hz": np.mean(np.log2(1 + s1)),
            "rate_opt_bps_hz": np.mean(np.log2(1 + best))}, loss1

## 3. ML method: MLP classifier (model selection on validation *routes*)

In [ ]:
def train_mlp(Xtr, ytr, Xva, yva, seed):
    sc = StandardScaler().fit(Xtr)                            # preprocessing fitted on training routes only
    Xtr_s, Xva_s = sc.transform(Xtr), sc.transform(Xva)
    clf = MLPClassifier(hidden_layer_sizes=(128, 128), batch_size=256, learning_rate_init=1e-3,
                        alpha=1e-4, random_state=seed)
    best, best_clf, hist = -1, None, []
    for ep in range(EPOCHS):                                  # one epoch per partial_fit call
        clf.partial_fit(Xtr_s, ytr, classes=np.arange(B))
        va = clf.score(Xva_s, yva)
        hist.append((clf.loss_, va))
        if va > best:
            best, best_clf = va, copy.deepcopy(clf)           # keep the epoch that is best on validation routes
    return sc, best_clf, np.array(hist)

def mlp_predict(sc, clf, X):
    p = clf.predict_proba(sc.transform(X))
    cand3 = np.argsort(-p, axis=1)[:, :3]
    return cand3[:, 0], cand3

## 4. Run the experiment (3 seeds) and collect results

In [ ]:
results, pooled, histories = [], [], []
t_start = time.time()
for seed in SEEDS:
    rng = np.random.default_rng(seed)
    D = simulate(N_ROUTES, rng)
    tr, va, te = make_splits(D["route"], rng, "route")
    # leakage check: no route may appear in two splits
    assert not (set(D["route"][tr]) & set(D["route"][va])) and not (set(D["route"][tr]) & set(D["route"][te])) \
        and not (set(D["route"][va]) & set(D["route"][te])), "route leakage!"
    sc, clf, hist = train_mlp(D["X"][tr], D["y"][tr], D["X"][va], D["y"][va], seed)
    histories.append(hist)

    X, y, snr = D["X"][te], D["y"][te], D["snr_true"][te]
    preds = {}
    t0 = time.time(); b1, b3 = baseline_strongest_rsrp(D["rsrp"][te]); t_base = (time.time()-t0)/len(y)*1e3
    preds["Baseline: strongest RSRP"] = (b1, b3, t_base, len(meas_idx))
    t0 = time.time(); m1, m3 = mlp_predict(sc, clf, X); t_mlp = (time.time()-t0)/len(y)*1e3
    preds["Proposed: MLP"] = (m1, m3, t_mlp, len(meas_idx))

    sbest_db = 10*np.log10(snr.max(1))
    for name, (p1, p3, ms, nb) in preds.items():
        r, loss = evaluate(p1, p3, snr, y)
        results.append({"seed": seed, "method": name, "beams_measured": nb, "ms_per_sample": ms, **r})
        pooled.append(pd.DataFrame({"seed": seed, "method": name, "correct": (p1 == y), "loss_dB": loss,
                                    "speed": D["speed"][te], "snr_bin_dB": sbest_db, "blocked": D["blocked"][te]}))
    print(f"seed {seed}: train {tr.sum()} / val {va.sum()} / test {te.sum()} samples "
          f"({len(set(D['route'][te]))} test routes) | MLP top-1 {preds['Proposed: MLP'][0].__eq__(y).mean():.3f}")
print(f"done in {time.time()-t_start:.0f}s")
res = pd.DataFrame(results); pool = pd.concat(pooled, ignore_index=True)

## 5. Main results table (mean ± std over seeds)

In [ ]:
cols = {"top1": "Top-1 acc", "top3": "Top-3 acc", "within1": "Within ±1 beam", "snr_loss_dB": "SNR loss (dB)",
        "snr_loss_top3_dB": "SNR loss, best of top-3 (dB)", "outage_%": "Outage >3 dB (%)",
        "rate_bps_hz": "Rate (bit/s/Hz)", "beams_measured": "Beams measured", "ms_per_sample": "Inference (ms/sample)"}
g = res.groupby("method")[list(cols)].agg(["mean", "std"])
order = ["Baseline: strongest RSRP", "Proposed: MLP"]
table = pd.DataFrame(index=order)
for c, label in cols.items():
    dec = 3 if c in ("top1", "top3", "within1") else (4 if c == "ms_per_sample" else 2)
    table[label] = [f"{g.loc[m, (c, 'mean')]:.{dec}f} ± {g.loc[m, (c, 'std')]:.{dec}f}" for m in order]
display(table.T)
print("Optimal-beam rate (upper bound): %.2f bit/s/Hz" % res["rate_opt_bps_hz"].mean())
table.T.to_csv("results_main_table.csv")

## 6. Results by operating condition (speed, SNR, blockage)

In [ ]:
two = pool[pool.method.isin(order)].copy()
two["Speed (m/s)"] = pd.cut(two.speed, [0, 5, 10, 15, 20, 25], labels=["1-5", "5-10", "10-15", "15-20", "20-25"])
two["Best-beam SNR (dB)"] = pd.cut(two.snr_bin_dB, [-100, 0, 10, 20, 100], labels=["<0", "0-10", "10-20", ">20"])
two["Link state"] = np.where(two.blocked, "LoS blocked", "LoS clear")

def cond_table(col):
    t = two.groupby([col, "method"], observed=True).agg(top1=("correct", "mean"), loss=("loss_dB", "mean"), n=("correct", "size"))
    t = t.unstack("method")
    out = pd.DataFrame({"n": t[("n", order[0])]})
    for m, short in zip(order, ["Base", "MLP"]):
        out[f"Top-1 {short}"] = t[("top1", m)].round(3)
        out[f"SNR loss {short} (dB)"] = t[("loss", m)].round(2)
    return out

for col in ["Speed (m/s)", "Best-beam SNR (dB)", "Link state"]:
    print(f"\n--- by {col} ---"); display(cond_table(col))

fig, ax = plt.subplots(1, 3, figsize=(17, 4.5))
for a, col in zip(ax, ["Speed (m/s)", "Best-beam SNR (dB)", "Link state"]):
    t = cond_table(col); x = np.arange(len(t)); w = 0.38
    a.bar(x - w/2, t["SNR loss Base (dB)"], w, label="Baseline: strongest RSRP")
    a.bar(x + w/2, t["SNR loss MLP (dB)"], w, label="Proposed: MLP")
    a.set_xticks(x); a.set_xticklabels(t.index, rotation=0); a.set(xlabel=col, ylabel="Mean SNR loss vs best beam (dB)")
    a.grid(alpha=.3, axis="y"); a.legend()
ax[0].set_title("SNR loss vs mobility"); ax[1].set_title("SNR loss vs SNR condition"); ax[2].set_title("SNR loss vs blockage")
plt.tight_layout(); plt.savefig("fig_conditions.png", dpi=150); plt.show()

## 7. Leakage check: random split vs route split

A random sample split puts neighbouring snapshots of the same route in both training and test sets. The accuracy looks better, but it is not what you would get on a new route.

In [ ]:
rng = np.random.default_rng(0)
D = simulate(N_ROUTES, rng)
rows = []
for mode in ["route", "random"]:
    tr, va, te = make_splits(D["route"], np.random.default_rng(0), mode)
    shared = len(set(D["route"][tr]) & set(D["route"][te]))
    sc, clf, _ = train_mlp(D["X"][tr], D["y"][tr], D["X"][va], D["y"][va], 0)
    p1, p3 = mlp_predict(sc, clf, D["X"][te])
    r, _ = evaluate(p1, p3, D["snr_true"][te], D["y"][te])
    rows.append({"split": mode, "routes shared by train and test": shared, "top-1": round(r["top1"], 3),
                 "SNR loss (dB)": round(r["snr_loss_dB"], 2)})
leak = pd.DataFrame(rows).set_index("split"); display(leak)

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].bar(leak.index, leak["top-1"], color=["tab:green", "tab:red"]); ax[0].set(ylabel="Top-1 accuracy", title="Leaky vs leakage-safe split")
ax[0].grid(alpha=.3, axis="y")
h = np.mean(histories, axis=0)
ax[1].plot(h[:, 1], label="validation top-1 (held-out routes)"); ax[1].set(xlabel="Epoch", title="MLP training (mean of seeds)")
ax2 = ax[1].twinx(); ax2.plot(h[:, 0], "C1", label="training loss"); ax2.set_ylabel("loss")
ax[1].legend(loc="center right"); plt.tight_layout(); plt.savefig("fig_leakage_training.png", dpi=150); plt.show()

## 8. What to write in the report (checklist from the slides)

* **Formulation:** scenario, signal model (LoS + 3 reflected paths, Rician/Rayleigh fading, blockage), input/output, objective (minimise SNR loss under a short inference time).
* **Features:** RSRP (dB) of the measured beams, noisy position (m), noisy velocity (m/s). All are available before transmission. Position noise std 1.5 m, velocity noise std 0.5 m/s. The "previous beam" from the slides is the strongest-RSRP beam of the last sweep, so it is already contained in the RSRP vector and is not added as a separate feature.
* **Data:** simulated (500 routes x 48 steps per seed, 3 seeds); labels are the best beam from the noiseless channel; SNR range set by the transmit power (0 to 30 dBm).
* **Split and leakage:** by complete route (60/20/20), scaler fitted on training routes, validation routes used for model selection, route overlap asserted to be empty, random-split demo in section 7.
* **Baseline vs MLP:** same inputs and same test routes; only the decision rule differs.
* **Results:** main table (section 5), per-condition tables and plots (section 6).
* **Limitations (be honest):**
  * the baseline is effectively "keep the beam that was strongest at the last sweep", so it suffers from both noise and staleness;
  * the environment (reflector positions) is the same in training and test, so there is no test of unseen deployments;
  * everything is simulated, with no real measurements;
  * the MLP needs positions, which may not be available indoors or with GPS errors.
* **Ideas to extend:** `STRIDE = 4` (partial sweep, much lower measurement cost), unseen scatterer layouts, larger position noise, random forest as another ML method.